## Notebook Description: Interactive Analysis of Methane Intensity and TDDM

This Colab notebook presents an interactive data analysis and visualization focusing on methane intensity, denoted as $\text{methane_intensity}$, and total digestible dry matter ($\text{tddm}$). It begins by loading two datasets, 'subsets_1_2_3' and 'control_grasses', which are then concatenated into a unified DataFrame. The primary objective is to visualize the relationship between these two variables, $\text{methane_intensity}$ and $\text{tddm}$, using an Altair scatterplot. The visualization incorporates interactive dropdown filters for 'subset' and 'functional_group', allowing dynamic exploration of the data. Conditional encoding is applied, utilizing distinct colors for different 'subset' categories and unique shapes for various 'functional_group' classifications, to highlight patterns and facilitate comparative analysis.

# Connect to google drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Import libraries

In [2]:
!pip install itables

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.0 MB/s eta 0:00:00


In [3]:
import pandas as pd
from itables import init_notebook_mode, show
from itables.sample_dfs import get_countries
import altair as alt

# Data load

In [4]:
subsets_1_2_3 = pd.read_csv('/content/drive/MyDrive/lmf/data/04_28_2026_new_dashboards_subsets_1_2_3_analysis_and_graphics/subsets_1_2_3_gas_dashboard_small.csv')
subsets_1_2_3.head(2)

,subset,id_lab,id,tax_name,functional_group,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,methane_intensity,tddm
0,1,F24-3469,ABC-10647,Stylosanthes scabra,Herbaceous_legumes,15.90,18.79,48.31,67.34
1,1,F24-3470,ABC-11194,Stylosanthes hamata,Herbaceous_legumes,15.17,18.02,46.12,59.25


In [5]:
control_grasses = pd.read_csv('/content/drive/MyDrive/lmf/data/04_28_2026_new_dashboards_subsets_1_2_3_analysis_and_graphics/control_grasses.csv')
control_grasses.head()

,subset,id_lab,id,tax_name,functional_group,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,methane_intensity,tddm
0,4,F26-1076,CIAT-BR06_423_Opt_F,Urochloa interespecifico,Grass,14.60,16.30,51.05,51.15
1,4,F26-1077,CIAT_MulatoII_Opt_D,Urochloa interespecifico,Grass,13.15,15.15,46.45,55.80


# Concat subsets data with controls

In [13]:
df = pd.concat([subsets_1_2_3, control_grasses])
df.tail()

Loading ITables v2.7.3 from the internet... (need help?)


In [16]:
grasses = df[df['functional_group'] == 'Grass']
grasses.count()

Loading ITables v2.7.3 from the internet... (need help?)


# Functional group counts

In [14]:
df.functional_group.value_counts()

Loading ITables v2.7.3 from the internet... (need help?)


# Interactive data frame

In [15]:
init_notebook_mode(all_interactive=True)

targets = [df.columns.get_loc("subset"), df.columns.get_loc("tax_name"), df.columns.get_loc("functional_group"),]

show(df, paging=True, pageLength=25, dom="Pfrtip",
     searchPanes={"cascadePanes": True,"layout": "columns-3"},
     columnDefs=[{"searchPanes": {"show": True},"targets": targets},
        {"searchPanes": {"show": False},"targets": "_all"}],
    buttons=["copyHtml5","csvHtml5","excelHtml5"])

/usr/local/lib/python3.12/dist-packages/itables/typing.py:302: SyntaxWarning: These arguments are not documented in ITableOptions: {'dom'}. You can silence this warning by setting `itables.options.warn_on_undocumented_option=False`. If you believe ITableOptions should be updated, please make a PR or open an issue at https://github.com/mwouts/itables
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/itables/typing.py:302: SyntaxWarning: These arguments are not documented in DTForITablesOptions: {'dom'}. You can silence this warning by setting `itables.options.warn_on_undocumented_option=False`. If you believe ITableOptions should be updated, please make a PR or open an issue at https://github.com/mwouts/itables
  warnings.warn(


Loading ITables v2.7.3 from the internet... (need help?)


# Interactive scatter plot

In [ ]:
# Create a selection for the 'subset' column
subset_selection = alt.selection_point(fields=['subset'], bind=alt.binding_select(options=['All'] + list(df['subset'].unique()), name='Select Subset '))

# Create a selection for the 'functional_group' column
functional_group_selection = alt.selection_point(fields=['functional_group'], bind=alt.binding_select(options=['All'] + list(df['functional_group'].unique()), name='Select Functional Group '))

In [ ]:
chart = alt.Chart(df).mark_circle(size=60).encode(
    x='tddm',
    y='methane_intensity',
    color=alt.condition(
        subset_selection,
        alt.Color('subset:N',
                  scale=alt.Scale(domain=[1, 2, 3, 4],
                                  range=['cornflowerblue', 'orange', 'lightcoral', 'red'])), # Distinct color for subset 4 (purple)
        alt.value('lightgray')
    ),
    shape=alt.condition(
        functional_group_selection,
        alt.Shape('functional_group:N',
                  scale=alt.Scale(domain=['Herbaceous_legumes', 'Grass', 'others', 'Tree_legumes'],
                                  range=['circle', 'square', 'triangle', 'diamond'])), # Distinct shapes for functional groups
        alt.value('circle')
    ),
    tooltip=[
        'id:N',
        'id_lab:N',
        'tax_name:N',
        'functional_group:N',
        'subset:N',
        'ch4_percentage_in_gas_8h:Q',
        'ch4_percentage_in_gas_24h:Q',
        'tddm:Q',
        'methane_intensity:Q'
    ]
).add_params(
    subset_selection,
    functional_group_selection
).transform_filter(
    subset_selection
).transform_filter(
    functional_group_selection
).properties(
    title='Methane Intensity vs. TDDM with Interactive Filters',
    width=800,
    height=600
).interactive().configure_title(
    fontSize=20
).configure_axis(
    labelFontSize=14,
    titleFontSize=16
).configure_legend(
    titleFontSize=14,
    labelFontSize=12
)

chart

alt.Chart(...)